In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from sklearn.model_selection import train_test_split

In [2]:
x, y = [], []
DATA_PATH = './dataset/'
img_row, img_col = 100, 100
counter = 0
classes = os.listdir(DATA_PATH)
print(classes) 

['Boot', 'Sandal', 'Shoe']


In [ ]:
for class_path in classes:
    for img_path in os.listdir(f"{DATA_PATH}{class_path}"):
        path = f"{DATA_PATH}{class_path}/{img_path}"
        label = counter
        img = keras.preprocessing.image.load_img(path, target_size=(img_row, img_col))
        img = keras.preprocessing.image.img_to_array(img)
        x.append(img)
        y.append(label)

    counter += 1

x = np.array(x)
y = np.array(y)
x = x.astype('float32') / 255.0

In [ ]:
x_train, y_train, x_temp, y_temp = train_test_split(
    x, y, test_size=0.2, random_state=42
)

x_test, y_test, x_val, y_val = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42
)

y_train = keras.utils.to_categorical(y_train)
y_test = keras.utils.to_categorical(y_test)
y_val = keras.utils.to_categorical(y_val)
print(x_train.shape)

In [ ]:
import keras.layers

model = keras.Sequential()

model.add(keras.layers.Conv2D(6, (6,6), activation="relu", image_size=(img_row, img_col, 3)))
model.add(keras.layers.AveragePooling2D((2,2), 2))
model.add(keras.layers.Conv2D(16, (6,6), activation="relu"))
model.add(keras.layers.AveragePooling2D((2,2), 2))

model.add(keras.layers.Flatten())

model.add(keras.layers.Dense(120, activation="relu"))
model.add(keras.layers.Dense(84, activation="relu"))
model.add(keras.layers.Dense(3, activation="softmax"))

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    x_train, y_train,
    batch_size=120, epochs=10,
    validation_data=(x_val, y_val)
)

In [ ]:
model.evaluate(x_test, y_test)

In [ ]:
prediction = model.predict(x_test)
predicted_label = np.argmax(prediction, 1)

for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(x_test[i], cmap='gray')
    plt.axis('off')
    plt.text(
        0.5, -0.15,
        classes[predicted_label[i]], fontsize=10, ha="center",
        transform=plt.gca().transAxes
    )

In [ ]:
plt.figure(figsize=(12, 8))

plt.subplot(1,2,1)
plt.plot(history.history["Accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history["Loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Loss Accuracy")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()